# Aspire 3 : Observabilite .NET moderne — Serilog, OpenTelemetry, ActivitySource

Ce notebook digere la Part 5 de la serie *The Unexpected AI Stack: C# + .NET*
(voir issue #11516) : la pile d'observabilite que l'auteur branche autour de son
agent — **Serilog** pour les logs structures, **OpenTelemetry** pour les traces,
et un point souvent neglige, l'instrumentation **call-site** via `ActivitySource`
et les attributs `Caller*`.

L'observabilite, c'est repondre a trois questions sur un systeme en production :
**que s'est-il passe ?** (logs), **ou exactement ?** (traces) et **combien ?**
(metrics). Les trois se rejoignent dans le standard OpenTelemetry (OTel), devenu
le lingua franca des fournisseurs de telemetry — instrumenter une fois, exporter
vers Jaeger, Grafana, Azure Monitor ou la console, sans changer le code metier.

Ce que ce notebook demontre, cellule par cellule :

1. le log structure Serilog — le template `{Propriete}` produit des champs, pas
   des strings interpolables ;
2. le span OpenTelemetry — un `ActivitySource` alimente un `TracerProvider`
   muni d'un exporteur console ;
3. l'instrumentation call-site — `CallerMemberName` / `CallerFilePath` /
   `CallerLineNumber` remplissent les tags `code.function` / `code.file.path` /
   `code.line.number` **automatiquement**, sans argument passe a la main ;
4. la correlation — les evenements Serilog et les spans OTel partagent les memes
   identifiants (`@tr` = TraceId, `@sp` = SpanId) : un log se rattache a sa trace
   sans aucune glue de notre cru.

**Position dans la serie** : Aspire 1 (orchestration GenAI) et Aspire 2 (stack
reel) couvrent l'AppHost ; le dossier `aspire-otel/` du repo contient un AppHost
OTLP de reference. Ce notebook se place en amont de l'outil : il execute la
**librairie** (.NET Interactive en local), pas l'orchestrateur.

## 1. Serilog : le log structure

Le defaut de `Console.WriteLine` et du `ILogger` basique : tout devient du texte.
Chercher « toutes les requetes de l'etudiant-42 » dans un log texte = grep
fragile sur des formats dates. Le log structure inverse le rapport : le message
est un **template nomme**, et chaque trou `{Utilisateur}` devient un **champ**
typé dans la sortie.

Avec le formateur compact JSON de Serilog, chaque evenement devient un objet —
`@t` (horodatage), `@mt` (template), `@tr`/`@sp` (trace/span, on y revient en
section 4), puis les champs nommes du template.

In [1]:
// Pin Serilog 4.0.0 : Sinks.Console 6.0.0 est compile contre 4.0.0 (et non 4.1.0).
#r "nuget: Serilog, 4.0.0"
#r "nuget: Serilog.Sinks.Console, 6.0.0"
#r "nuget: Serilog.Formatting.Compact, 3.0.0"
#r "nuget: OpenTelemetry, 1.11.1"
#r "nuget: OpenTelemetry.Exporter.Console, 1.11.1"
using Serilog;
using System.Diagnostics;
using System.IO;
using System.Threading;
using OpenTelemetry;
using OpenTelemetry.Exporter;
using OpenTelemetry.Trace;

Console.WriteLine("packages charges");

The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages OpenTelemetry, 1.11.1 OpenTelemetry.Exporter.Console, 1.11.1 Serilog, 4.0.0 Serilog.Formatting.Compact, 3.0.0 Serilog.Sinks.Console, 6.0.0

packages charges


In [2]:
// Le logger Serilog : template nomme, sink console, formateur JSON compact.
var log = new LoggerConfiguration()
    .Enrich.WithProperty("app", "aspire03-demo")
    .WriteTo.Console(new Serilog.Formatting.Compact.CompactJsonFormatter())
    .CreateLogger();

// Template structure : {Utilisateur} et {Question} deviennent des CHAMPS.
log.Information("requete recue {Utilisateur} question={Question}",
    "etudiant-42", "qu'est-ce qu'un span ?");

// A comparer : l'interpolation C# classique produit une ligne de texte plate.
Console.WriteLine($"requete recue utilisateur=etudiant-42 (interpolation, pas de champs)");

{"@t":"2026-08-17T20:54:22.6994136Z","@mt":"requete recue {Utilisateur} question={Question}","@tr":"b41a1debaa5b9b2e49de20d7001be7dc","@sp":"3850730a4ab65fdb","Utilisateur":"etudiant-42","Question":"qu'est-ce qu'un span ?","app":"aspire03-demo"}


requete recue utilisateur=etudiant-42 (interpolation, pas de champs)


**Lecture du resultat.** Deux lignes pour la meme information : la ligne
Serilog est un objet JSON (`@t`, `@mt`, puis `Utilisateur` et `Question` comme
champs distincts) ; la ligne interpolee est du texte. Dans un agrégateur
(Elastic, Loki, Application Insights), la premiere se filtre par
`Utilisateur = "etudiant-42"` en index ; la seconde exige un grep. C'est tout
l'enjeu du log structure : **la machine indexe, l'humain relit**.

Le prefixe `app=aspire03-demo` vient de `Enrich.WithProperty` — un champ pose
une fois pour toutes les lignes du logger (environnement, version, service).

## 2. OpenTelemetry : le span

Cote traces, l'unite est le **span** : une duree nommee, avec debut, fin, tags
et relation parent/enfant. En .NET, un span s'appelle un `Activity` (l'API
existe dans le BCL), et c'est `ActivitySource` qui les emet — un producteur
nomme que le SDK OTel branche ensuite vers des exporteurs.

La plomberie complete tient en quelques lignes : creer la source, construire un
`TracerProvider` qui l'ecoute, y ajouter un exporteur. Ici l'exporteur est la
console ; en production ce serait OTLP vers un collector.

Deux lignes du code ci-dessous sont dictees par le contexte **kernel** et
commentees comme telles : le sampler explicite et le processeur d'export
immediat. Dans un programme console ordinaire, `AddConsoleExporter()` seul
suffirait.

In [3]:
// L'ActivitySource : le producteur de spans, nomme comme le service.
// (static dans un programme ; ici variable simple car cellule notebook.)
var source = new ActivitySource("Aspire03.Demo");

// Le TracerProvider OTel : ecoute la source, exporte sur la console.
// Deux precautions dictees par le contexte kernel (verifiees empiriquement) :
// - SetSampler(new AlwaysOnSampler()) : le kernel .NET Interactive maintient
//   une Activity parent ambiante, et le sampler par defaut (ParentBased)
//   refuserait tout span local -> StartActivity retournerait null ;
// - SimpleActivityExportProcessor : exporte chaque span a sa fermeture ; le
//   processeur batch par defaut ne flush qu'a la destruction du provider,
//   qui n'arrive jamais dans un kernel vivant.
var tracer = Sdk.CreateTracerProviderBuilder()
    .AddSource("Aspire03.Demo")
    .SetSampler(new AlwaysOnSampler())
    .AddProcessor(new SimpleActivityExportProcessor(new ConsoleActivityExporter(new ConsoleExporterOptions())))
    .Build();

using (var racine = source.StartActivity("TraitementAgent"))
{
    racine?.SetTag("run.id", Environment.TickCount64);
    using (var enfant = source.StartActivity("AppelModele"))
    {
        enfant?.SetTag("model", "factice-local");
        Thread.Sleep(50);  // simule l'appel
    }
}
Console.WriteLine("spans exportes ci-dessus");

Activity.TraceId:            b41a1debaa5b9b2e49de20d7001be7dc


Activity.SpanId:             6ba00c1ebd281043


Activity.TraceFlags:         Recorded


Activity.ParentSpanId:       1c2bbfd49cf6a1d3


Activity.DisplayName:        AppelModele


...

    telemetry.sdk.name: opentelemetry


    telemetry.sdk.language: dotnet


    telemetry.sdk.version: 1.11.1


    service.name: unknown_service:dotnet-interactive


spans exportes ci-dessus


**Lecture des spans.** L'exporteur console dechiffre chaque `Activity` :
identifiants (`TraceId`, `SpanId`, `ParentSpanId` — l'enfant pointe vers le
parent), `DisplayName` (le nom passe a `StartActivity`), `Kind`, `StartTime`,
`Duration`, puis les **Tags** poses a la main (`run.id`, `model`) et la
`Resource` — les attributs du process exporteur (`telemetry.sdk.*`,
`service.name`).

L'ordre d'affichage surprend au premier regard : **l'enfant `AppelModele`
apparait avant le parent `TraitementAgent`**. Un span n'est exporte qu'a sa
fermeture, et l'enfant se ferme le premier — l'ordre d'export est l'ordre des
fermetures, pas celui des ouvertures. La hierarchie, elle, se lit dans les
`ParentSpanId`.

Deux details qui comptent : le `TraitementAgent` racine porte lui-meme un
`ParentSpanId` — c'est l'activite ambiante du kernel .NET Interactive qui
execute la cellule (raison pour laquelle le sampler explicite ci-dessus est
necessaire) ; et `service.name` vaut `unknown_service:dotnet-interactive`
faute de variable d'environnement `OTEL_SERVICE_NAME` — la poser est la
premiere configuration d'un deploiement reel, ou elle identifierait le service
dans le backend de traces.

## 3. L'instrumentation call-site : les tags qui ne mentent jamais

Un piege classique de l'instrumentation manuelle : ecrire le nom de la fonction
**a la main** dans le tag — `SetTag("code.function", "Traiter")` — puis
renommer la fonction et laisser le tag mentir. Les attributs de compilation
`[CallerMemberName]`, `[CallerFilePath]`, `[CallerLineNumber]` ferment ce piege :
le compilateur injecte la valeur **reelle** au site d'appel, gratuitement.

Le helper ci-dessous est le pattern de la Part 5 : un demarrage de span qui
porte son propre code de localisation. Les tags produits suivent la convention
OTel semantique (`code.function`, `code.file.path`, `code.line.number`).

In [4]:
#nullable enable
// Helper call-site : le compilateur remplit member/file/line au site d'appel.
Activity? SpanCallSite(
    [System.Runtime.CompilerServices.CallerMemberName] string member = "",
    [System.Runtime.CompilerServices.CallerFilePath] string file = "",
    [System.Runtime.CompilerServices.CallerLineNumber] int line = 0)
{
    var a = source.StartActivity(member);
    a?.SetTag("code.function", member);
    a?.SetTag("code.file.path", Path.GetFileName(file));
    a?.SetTag("code.line.number", line);
    return a;
}

// La preuve : les tags portent le VRAI nom, fichier et numero de ligne.
using (var s = SpanCallSite())
{
    Console.WriteLine("span ouvert, tags poses par le compilateur");
}

span ouvert, tags poses par le compilateur


Activity.TraceId:            b41a1debaa5b9b2e49de20d7001be7dc


Activity.SpanId:             448e454029206ffb


Activity.TraceFlags:         Recorded


Activity.ParentSpanId:       36c1ccb5f591d19f


...

Resource associated with Activity:


    telemetry.sdk.name: opentelemetry


    telemetry.sdk.language: dotnet


    telemetry.sdk.version: 1.11.1


    service.name: unknown_service:dotnet-interactive


**Lecture du span affiche.** Regardez les trois tags : `code.function` porte
`<Initialize>` et `code.line.number` vaut 15 — la ligne de l'appel. Aucune de
ces valeurs n'a ete ecrite a la main : le compilateur C# les injecte au site
d'appel via les attributs `Caller*`.

Deux nuances propres au notebook, lisibles dans les tags : `<Initialize>` est
le nom que le kernel donne au bloc racine d'une cellule (chaque cellule est
compilee comme une submission Roslyn) — c'est precisement la preuve que la
valeur vient du compilateur et non d'une saisie ; et `code.file.path` est vide
parce qu'une cellule n'a pas de fichier source sur disque. Dans un programme
reel, le membre serait une methode nommee et le chemin son fichier source —
la section 4 le montre : le meme helper, appele depuis la methode `Traiter`,
porte `code.function: Traiter`.

## 4. La correlation : un log dans sa trace

Le morceau de glue que la pile moderne offre gratuitement : quand un evenement
Serilog est emis **pendant qu'un span est actif**, l'enrichisseur de contexte
pose automatiquement `@tr` (TraceId) et `@sp` (SpanId) dans le JSON — les
memes identifiants que l'exporteur de traces. Resultat : cliquer sur un log
dans un dashboard et retomber sur le span exact qui l'entourait.

La demo : un mini-agent factice dont chaque etape ouvre un span (via le helper
call-site) ET emet un log Serilog structure.

In [5]:
// Mini-agent factice : chaque etape = un span call-site + un log Serilog.
string Traiter(string utilisateur, string question)
{
    using var span = SpanCallSite();
    log.Information("requete recue {Utilisateur} question={Question}", utilisateur, question);
    var duree = question.Length * 7;
    Thread.Sleep(30);
    log.Information("reponse emise duree_ms={DureeMs}", duree);
    span?.SetTag("reponse.duree_ms", duree);
    return $"reponse en {duree} ms";
}

using (var racine = source.StartActivity("Run"))
{
    Traiter("etudiant-42", "qu'est-ce qu'un span ?");
    Traiter("etudiant-07", "pourquoi CallerMemberName ?");
}

{"@t":"2026-08-17T20:54:23.3744106Z","@mt":"requete recue {Utilisateur} question={Question}","@tr":"b41a1debaa5b9b2e49de20d7001be7dc","@sp":"23c2f17d5e5e593c","Utilisateur":"etudiant-42","Question":"qu'est-ce qu'un span ?","app":"aspire03-demo"}


{"@t":"2026-08-17T20:54:23.4087194Z","@mt":"reponse emise duree_ms={DureeMs}","@tr":"b41a1debaa5b9b2e49de20d7001be7dc","@sp":"23c2f17d5e5e593c","DureeMs":154,"app":"aspire03-demo"}


Activity.TraceId:            b41a1debaa5b9b2e49de20d7001be7dc


Activity.SpanId:             23c2f17d5e5e593c


Activity.TraceFlags:         Recorded


...

Resource associated with Activity:


    telemetry.sdk.name: opentelemetry


    telemetry.sdk.language: dotnet


    telemetry.sdk.version: 1.11.1


    service.name: unknown_service:dotnet-interactive


**Lecture de la correlation — la demonstration cles de ce notebook.** Relevez
le `@tr` des evenements Serilog et le `TraceId` des spans exportes : **meme
valeur**. Le `@sp` de « requete recue » egale le `SpanId` du span `Traiter`
(emis pendant qu'il est actif). Aucune ligne de code ne relie les deux fleuves :
Serilog lit l'`Activity.Current` du thread, OTel exporte le meme. C'est ce qui
permet, dans un backend type Jaeger ou Application Insights, de partir d'une
erreur dans les logs et de reconstituer toute la chronologie du traitement.

En production, la plomberie ne change pas : seul l'exporteur passe de la
console a OTLP (`AddOtlpExporter`), et le dashboard Aspire (port 15050 par
defaut) ou `aspire otel spans` consomme le meme flux.

## 5. Exercices

Les trois exercices font varier un levier a la fois : un **tag metier**
(exercice 1), un **enrichissement global** (exercice 2), une **metrique**
(exercice 3 — le troisieme pilier, volontairement laisse de cote jusqu'ici).

### Exercice 1 : span metier chiffre

Ecrire une fonction `MesurerReponse` qui ouvre un span via `SpanCallSite`,
interroge un modele **factice** (un `Thread.Sleep` de duree aleatoire fait
l'affaire), puis pose le tag `reponse.tokens` avec un nombre entier. Verifier
dans l'export console que le tag apparait sur le span.

In [6]:
// Exercice 1 : a completer
// Indice : Random.Shared.Next(50, 200) pour la duree ; question.Length * 3 pour les tokens.
// Etape 1 : ouvrir le span avec SpanCallSite()
// Etape 2 : simuler l'appel (Thread.Sleep)
// Etape 3 : poser le tag reponse.tokens puis retourner la duree
Console.WriteLine("Exercice 1 a completer");

Exercice 1 a completer


### Exercice 2 : champ global d'environnement

Ajouter au logger un enrichissement d'environnement (par exemple
`env = "notebook"`) de sorte que **toutes** les lignes posterieures portent le
champ. Emettre deux evenements et verifier qu'ils portent tous les deux le
champ sans qu'on le re-ecrive dans le template.

In [7]:
// Exercice 2 : a completer
// Indice : repartir de new LoggerConfiguration(), chainer .Enrich.WithProperty("env", "notebook")
// Etape 1 : construire log2 avec l'enrichissement supplementaire (garder app=aspire03-demo)
// Etape 2 : emettre deux Information() avec des templates differents
// Etape 3 : lire les deux lignes et verifier que env est present partout
Console.WriteLine("Exercice 2 a completer");

Exercice 2 a completer


### Exercice 3 : la metrique, troisieme pilier

Logs et traces couverts — reste **metrics**. Creer un `Meter` nomme
`Aspire03.Metrics` avec un `Counter<int>` appele `questions_posees`,
l'incrémenter deux fois avec un tag `utilisateur` different, et construire un
`MeterProvider` console pour l'observer.

In [8]:
// Exercice 3 : a completer
// Indice : new Meter("Aspire03.Metrics") puis CreateCounter<int>("questions_posees")
// Etape 1 : creer le Meter et le Counter
// Etape 2 : Add(1, new KeyValuePair<string, object?>("utilisateur", "etudiant-42")) x2 utilisateurs
// Etape 3 : var mp = Sdk.CreateMeterProviderBuilder().AddMeter("Aspire03.Metrics").AddConsoleExporter().Build();
//           puis mp.ForceFlush() pour declencher l'export (meme raison que le Simple processor des traces)
Console.WriteLine("Exercice 3 a completer");

Exercice 3 a completer


## Conclusion

Ce notebook a execute le vrai stack d'observabilite .NET moderne, sans
orchestrateur :

- **Serilog** (A8) : templates structures, enrichissement global, formateur
  compact JSON — les logs sont des objets indexables ;
- **OpenTelemetry** (A9) : `ActivitySource` + `TracerProvider` + exporteur
  console — les traces avec identifiants, hierarchie et durees ;
- **ActivitySource call-site** (A10) : le helper `Caller*` pose les tags
  `code.function` / `code.file.path` / `code.line.number` injectes par le
  compilateur — l'instrumentation ne peut plus mentir sur sa position ;
- **la correlation** : `@tr`/`@sp` Serilog = TraceId/SpanId des spans, sans
  glue — la demonstration cles de la section 4.

La marche suivante est l'orchestrateur : dans une AppHost Aspire, l'exporteur
devient OTLP, le dashboard (port 15050) et la CLI `aspire otel spans` remplacent
l'export console — voir le dossier `aspire-otel/SkOtel.AppHost` du repo et le
notebook Aspire 2 pour le branchement d'un stack reel.

---
**Navigation** : [<< Aspire 2 GenAiStack Reel](02-Aspire-GenAiStack-Reel.ipynb) | [README Aspire](README.md)